# XGBoost on scale-free price / water-value features

Final model behind `submissions/2026-10-15_xgb_relctx_nolevel_cv0.9870.csv` (Kaggle public leaderboard **0.99106**).

**Task.** For every scheduling case (one week, 168 hours) predict for each of the 14 generating units the probability that it is ON
in each hour. Metric: micro-average ROC-AUC over all cases × units × hours.

**Shape of the problem.** We model one row per (case, hour) with 14 binary targets (one per unit): 2922 cases × 168 h = 490 896 rows.
`XGBClassifier` with a 2-D target fits one booster per unit on the same feature table.

**The idea in one sentence.** A unit runs when the electricity price is high relative to the *water value* (the marginal value of
keeping the water) of the reservoir that feeds its plant. Price and water value jump by an order of magnitude between years,
so we express that comparison in *scale-free* form (ratios, margin divided by the week's price spread) and drop the absolute levels.

**Pipeline.**

1. Base features from the kernel data: calendar, hourly price and weekly price statistics, hourly inflows, reservoir volume at case start, water values at start and end of the case.
2. Margin features: price minus the upstream reservoir's water value, per plant (topology from `data/extended`).
3. Constraint features: hourly minimum river flows and minimum reservoir volumes.
4. Scale-free features: price / water value, margin / weekly price spread, share of profitable hours, reservoir fill fraction and inflow volume. Then **drop every absolute price, water-value and margin column**.
5. Temporal context of the scale-free margin: rolling mean / max / min over 3, 6, 12 h, profitable hours per day, run lengths of profitable hours behind and ahead of the hour.
6. Honest time-aware validation: number of boosting rounds chosen by early stopping on 2021, score reported on 2022, final model refit on all years.

All reusable code lives in `src/`; the notebook only composes it. Feature building is one function applied identically to train and test.

## 0. Setup

### Shared code

Data loading, feature construction and the honest validation protocol. The same code is repeated at the top of every notebook so each one runs on its own.

In [ ]:
from pathlib import Path
from scipy.ndimage import uniform_filter1d, maximum_filter1d, minimum_filter1d
from sklearn.metrics import roc_auc_score
import json
import numpy as np
import pandas as pd
import re
import time
import xgboost as xgb

# ---- formerly src/train_xgb_baseline.py ----
# XGBoost baseline for the SINTEF unit-commitment task.
#
# Script version of xgboost_baseline.ipynb so it can run headless on the GPU desktop.
#
# Validation protocol: rounds are chosen by early stopping on 2021 (training on cases before 2021);
# the reported score is then from a model trained on cases before 2022 and scored on 2022, so the
# reported year never influences a modelling decision. A one-week gap keeps overlapping cases apart.
#
#     py train_xgb_baseline.py --device cuda          # GPU
#     python train_xgb_baseline.py --device cpu       # laptop
#
# Outputs (in --out, default ./results):
#     submission_xgb_baseline.csv   Kaggle submission (test cases 2023-2024)
#     val_pred_2022.npy             hold-out probabilities, shape (365*168, 14)
#     metrics.json                  hold-out micro AUC, per-unit AUC, best iteration, timings
#     feature_importance.csv        gain per feature (summed over the 14 boosters)

H = 168
SEED = 42

def load(data_dir):
    data_dir = Path(data_dir)
    uc = pd.read_csv(
        data_dir / "Unit_commitment_decisions.csv", parse_dates=["starttime"]
    )
    price = (
        pd.read_csv(
            data_dir / "Historical_day_ahead_price_2015_2025.csv", parse_dates=["date"]
        )
        .set_index("date")["Day-ahead price (EUR/MWh)"]
        .resample("h")
        .mean()
    )
    inflow = pd.read_csv(
        data_dir / "Historical_inflow_1958_2025.csv", parse_dates=["date"]
    ).set_index("date")
    inflow = inflow[inflow.index >= "2014-12-01"].resample("h").mean()
    volume = pd.read_csv(
        data_dir / "Historical_volume_2015_2024.csv", parse_dates=["date"]
    ).set_index("date")
    water_value = pd.read_csv(
        data_dir / "Synthetic_water_value_2015_2024.csv", parse_dates=["date"]
    ).set_index("date")
    return uc, price, inflow, volume, water_value

class FeatureBuilder:
    def __init__(self, price, inflow, volume, water_value):
        self.price, self.inflow, self.volume, self.wv = (
            price,
            inflow,
            volume,
            water_value,
        )

    @staticmethod
    def window(hourly_df, starts, horizon=H):
        idx = hourly_df.index.get_indexer(starts)
        assert (idx >= 0).all(), (
            "some case start times are missing from the hourly series"
        )
        pos = idx[:, None] + np.arange(horizon)[None, :]
        return hourly_df.to_numpy(dtype=np.float32)[pos]

    def __call__(self, starts):
        starts = pd.DatetimeIndex(starts)
        if starts.tz is None:
            starts = starts.tz_localize("UTC")
        n = len(starts)
        P = self.window(self.price.to_frame(), starts)[:, :, 0]
        I = self.window(self.inflow, starts)
        V = self.volume.reindex(starts).to_numpy(dtype=np.float32)
        W = self.wv.reindex(starts).to_numpy(dtype=np.float32)
        W_end = self.wv.reindex(starts + pd.Timedelta(days=7)).to_numpy(
            dtype=np.float32
        )
        hours = pd.DatetimeIndex(
            (
                starts.values[:, None] + np.arange(H)[None, :] * np.timedelta64(1, "h")
            ).ravel()
        )
        rep = lambda a: np.repeat(a, H)

        f = pd.DataFrame(
            {
                "t": np.tile(np.arange(H), n),
                "hour": hours.hour,
                "dow": hours.dayofweek,
                "month": hours.month,
                "doy": hours.dayofyear,
                "price": P.ravel(),
                "price_week_mean": rep(P.mean(1)),
                "price_week_std": rep(P.std(1)),
                "price_week_min": rep(P.min(1)),
                "price_week_max": rep(P.max(1)),
            }
        )
        f["price_rel"] = f["price"] - f["price_week_mean"]
        f["price_rank"] = (P.argsort(1).argsort(1) / (H - 1)).ravel()
        f["price_day_mean"] = np.repeat(P.reshape(n, 7, 24).mean(2), 24, axis=1).ravel()
        f["price_prev"] = np.concatenate([P[:, :1], P[:, :-1]], axis=1).ravel()   # edge-padded, no wrap
        f["price_next"] = np.concatenate([P[:, 1:], P[:, -1:]], axis=1).ravel()
        for j, c in enumerate(self.inflow.columns):
            f[f"inflow_{c}"] = I[:, :, j].ravel()
        f["inflow_total"] = I.sum(2).ravel()
        for j, c in enumerate(self.volume.columns):
            f[f"vol_{c}"] = rep(V[:, j])
        for j, c in enumerate(self.wv.columns):
            f[f"wv_{c}"] = rep(W[:, j])
        for j, c in enumerate(self.wv.columns):
            f[f"wv_end_{c}"] = rep(W_end[:, j])
        return f

def split_masks(starts, val_year, gap_days=7):
    """Case-level masks. train: cases whose whole week ends before val_year (so no hour is shared with the
    validation cases); val: cases starting in val_year. Returned per case, use np.repeat(mask, H) for rows."""
    starts = pd.DatetimeIndex(starts)
    val_start = pd.Timestamp(f"{val_year}-01-01")
    train = np.asarray(starts < val_start - pd.Timedelta(days=gap_days - 1))
    val = np.asarray(starts.year == val_year)
    return train, val

def honest_fit(X, Y, starts, params, tune_year=2021, report_year=2022, device="cuda", verbose=False, max_rounds=3000):
    """Two-stage validation so the reported year is never used to make a modelling decision.

    stage 1: train on cases before tune_year, early-stop on tune_year  -> number of rounds
    stage 2: train on cases before report_year with that many rounds   -> score on report_year
    Returns a dict with n_rounds, tune_auc, report_auc, per-unit AUC, the stage-2 model and its predictions.
    """
    common = dict(tree_method="hist", device=device, random_state=SEED, **params)
    tr1, va1 = (np.repeat(m, H) for m in split_masks(starts, tune_year))
    m1 = xgb.XGBClassifier(n_estimators=max_rounds, eval_metric="auc", early_stopping_rounds=100, **common)
    m1.fit(X[tr1], Y[tr1], eval_set=[(X[va1], Y[va1])], verbose=verbose)
    n_rounds = int(m1.best_iteration) + 1
    tune_auc = roc_auc_score(Y[va1], m1.predict_proba(X[va1]), average="micro")

    tr2, va2 = (np.repeat(m, H) for m in split_masks(starts, report_year))
    m2 = xgb.XGBClassifier(n_estimators=n_rounds, **common).fit(X[tr2], Y[tr2])
    pred = m2.predict_proba(X[va2])
    report_auc = roc_auc_score(Y[va2], pred, average="micro")   # identical to kaggle_metric.score
    per_unit = [roc_auc_score(Y[va2][:, j], pred[:, j]) for j in range(Y.shape[1])]
    return dict(n_rounds=n_rounds, tune_auc=float(tune_auc), report_auc=float(report_auc),
                per_unit=per_unit, model=m2, pred=pred, val_mask=va2)

# ---- formerly src/features_margin.py ----
# Price-minus-water-value features (see xgboost_price_minus_wv.ipynb).
# plant -> reservoir directly upstream, from the connections in data/extended/Tokke_Vinje_topology.yaml
UPSTREAM = {
    "Hogga": "Bandak", "Tokke": "Vinjevatn", "Lio": "Byrtevatn", "Byrte": "Botnedalsvatn",
    "Vinje": "Vaamarvatn", "Haukeli": "Vatjern", "Songa": "Songavatn", "Kjela": "Foersvatn",
    "Vesle Kjela": "Kjelavatn",
}

def add_margin_features(f):
    f = f.copy()
    for plant, res in UPSTREAM.items():
        f[f"margin_{plant}"] = f["price"] - f[f"wv_{res}"]
        f[f"margin_end_{plant}"] = f["price"] - f[f"wv_end_{res}"]
        f[f"margin_end_wk_{plant}"] = f["price_week_mean"] - f[f"wv_end_{res}"]
    return f

# ---- formerly src/features_constraints.py ----
# Environmental-constraint features from data/extended (see xgboost_constraints.ipynb).
#
# Hourly minimum river flows (6 rivers) and minimum reservoir volumes (3 reservoirs), plus the distance between the
# reservoir's volume at case start and its minimum. All are inputs the optimiser receives for the case week.

H = 168

def load_constraints(data_dir="data/extended"):
    data_dir = Path(data_dir)
    minflow = pd.read_csv(data_dir / "Constraint_min_flow.csv", parse_dates=["date"]).set_index("date").resample("h").mean()
    minvol = pd.read_csv(data_dir / "Constraint_min_volume.csv", parse_dates=["date"]).set_index("date").resample("h").mean()
    return minflow, minvol

class ConstraintFeatures:
    def __init__(self, minflow, minvol, volume):
        self.minflow, self.minvol, self.volume = minflow, minvol, volume

    @staticmethod
    def window(hourly_df, starts):
        idx = hourly_df.index.get_indexer(starts)
        assert (idx >= 0).all(), "case start missing from constraint series"
        return hourly_df.to_numpy(dtype=np.float32)[idx[:, None] + np.arange(H)[None, :]]   # (n, H, cols)

    def __call__(self, starts):
        starts = pd.DatetimeIndex(starts)
        if starts.tz is None:
            starts = starts.tz_localize("UTC")
        n = len(starts)
        rep = lambda a: np.repeat(a, H)
        MF = self.window(self.minflow, starts)
        MV = self.window(self.minvol, starts)
        V = self.volume.reindex(starts)

        f = {}
        for j, river in enumerate(self.minflow.columns):
            f[f"minflow_{river}"] = MF[:, :, j].ravel()                 # requirement at hour t
            f[f"minflow_{river}_wkmax"] = rep(MF[:, :, j].max(1))       # tightest requirement during the week
        for j, res in enumerate(self.minvol.columns):
            v0 = V[res].to_numpy(dtype=np.float32)
            f[f"minvol_{res}"] = MV[:, :, j].ravel()
            f[f"volgap_{res}"] = rep(v0) - MV[:, :, j].ravel()           # volume at case start minus minimum at hour t
            f[f"volgap_{res}_wkmin"] = rep(v0 - MV[:, :, j].max(1))      # worst-case slack during the week
        return pd.DataFrame(f)

def add_constraint_features(f, starts, constraint_features):
    extra = constraint_features(starts)
    assert len(extra) == len(f)
    return pd.concat([f.reset_index(drop=True), extra], axis=1)

# ---- formerly src/features_context.py ----
# Temporal-context and cascade features on top of the margin features (see xgboost_context.ipynb).
#
# Context (per plant, from margin_end_<plant> = price - end-of-week water value of the upstream reservoir):
#   centred rolling mean / max / min over 3, 6 and 12 hours, number of positive-margin hours in the same day,
#   rank of the margin within the day, and run lengths: consecutive positive (negative) margin hours up to t and from t on.
# Cascade (per plant with a downstream reservoir): the downstream reservoir's water value at case start and end,
#   the upstream-minus-downstream difference, and price minus that difference ("cascade margin").

H = 168
# plant -> reservoir the plant discharges into (plant -> reservoir connections in Tokke_Vinje_topology.yaml)
DOWNSTREAM = {"Tokke": "Bandak", "Vinje": "Vinjevatn", "Haukeli": "Vinjevatn", "Byrte": "Byrtevatn",
              "Kjela": "Hyljelihyl", "Vesle Kjela": "Foersvatn", "Songa": "Totak"}   # Lio -> river, Hogga -> outlet
WINDOWS = (3, 6, 12)

def _run_lengths(pos):
    """pos: bool (n, H). Returns consecutive-True counts ending at t and starting at t."""
    n = pos.shape[0]
    back = np.zeros((n, H), dtype=np.float32); ahead = np.zeros((n, H), dtype=np.float32)
    for t in range(H):
        back[:, t] = np.where(pos[:, t], (back[:, t - 1] if t else 0) + 1, 0)
    for t in range(H - 1, -1, -1):
        ahead[:, t] = np.where(pos[:, t], (ahead[:, t + 1] if t < H - 1 else 0) + 1, 0)
    return back, ahead

def add_context_features(f, col="margin_end", prefix="ctx", zero=0.0, windows=WINDOWS):
    """Context of f"{col}_{plant}" (default: the end-of-week margin). `zero` is the ON/OFF threshold of that signal
    (0 for a margin, 1 for a price/water-value ratio)."""
    n = len(f) // H
    out = {}
    for plant in UPSTREAM:
        M = f[f"{col}_{plant}"].to_numpy(dtype=np.float32).reshape(n, H)
        for w in windows:
            out[f"{prefix}_mean{w}_{plant}"] = uniform_filter1d(M, size=w, axis=1, mode="nearest").ravel()
            out[f"{prefix}_max{w}_{plant}"] = maximum_filter1d(M, size=w, axis=1, mode="nearest").ravel()
            out[f"{prefix}_min{w}_{plant}"] = minimum_filter1d(M, size=w, axis=1, mode="nearest").ravel()
        day = M.reshape(n, 7, 24)
        out[f"{prefix}_poshours_day_{plant}"] = np.repeat((day > zero).sum(2), 24, axis=1).ravel()
        out[f"{prefix}_rank_day_{plant}"] = (day.argsort(2).argsort(2) / 23.0).reshape(n, H).ravel()
        pos_back, pos_ahead = _run_lengths(M > zero)
        neg_back, neg_ahead = _run_lengths(M <= zero)
        out[f"{prefix}_posrun_back_{plant}"] = pos_back.ravel(); out[f"{prefix}_posrun_ahead_{plant}"] = pos_ahead.ravel()
        out[f"{prefix}_negrun_back_{plant}"] = neg_back.ravel(); out[f"{prefix}_negrun_ahead_{plant}"] = neg_ahead.ravel()
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)

def add_cascade_features(f):
    out = {}
    for plant, down in DOWNSTREAM.items():
        up = UPSTREAM[plant]
        out[f"casc_wv_down_{plant}"] = f[f"wv_{down}"].to_numpy()
        out[f"casc_wv_down_end_{plant}"] = f[f"wv_end_{down}"].to_numpy()
        diff_end = f[f"wv_end_{up}"].to_numpy() - f[f"wv_end_{down}"].to_numpy()
        out[f"casc_wvdiff_end_{plant}"] = diff_end
        out[f"casc_margin_{plant}"] = f["price"].to_numpy() - diff_end
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)

# ---- formerly src/features_relative.py ----
# Scale-free versions of the margin signal (Feature_Engineering slides: pairwise transforms x./y, x-y).
#
# Why: prices and water values shift by an order of magnitude between years (2020 mean price 9 EUR, 2022 mean 211 EUR),
# so a margin of +2 EUR means "clearly profitable" in 2020 but "noise" in 2022. Trees split on absolute thresholds and
# transfer badly across regimes (rolling validation: 2020 report AUC 0.92 vs 0.98 in 2021/2022). Ratios and margins
# measured in units of the week's own price spread do not depend on the price level.

H = 168

def add_relative_features(f):
    n = len(f) // H
    price = f["price"].to_numpy(dtype=np.float32)
    spread = f["price_week_std"].to_numpy(dtype=np.float32) + 1.0
    out = {}
    for plant, res in UPSTREAM.items():
        wv0 = np.maximum(f[f"wv_{res}"].to_numpy(dtype=np.float32), 1.0)         # floor at 1 EUR: wv ~0 in 2020
        wv1 = np.maximum(f[f"wv_end_{res}"].to_numpy(dtype=np.float32), 1.0)
        margin_end = f[f"margin_end_{plant}"].to_numpy(dtype=np.float32)
        out[f"ratio_start_{plant}"] = price / wv0                                   # price / water value
        out[f"ratio_end_{plant}"] = price / wv1
        out[f"ratio_wk_{plant}"] = f["price_week_mean"].to_numpy(dtype=np.float32) / wv1
        out[f"margin_end_rel_{plant}"] = margin_end / spread                        # margin in units of weekly price spread
        out[f"poshours_wk_{plant}"] = np.repeat((margin_end.reshape(n, H) > 0).mean(1), H)   # share of profitable hours
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)

# Absolute price / water-value levels. Dropping them forces the model onto scale-free features only.
LEVEL_PREFIXES = ("price", "wv_", "margin_")

def level_columns(f):
    keep = {"price_rank", "price_week_std"}   # already scale-free (rank) or needed as a spread measure
    return [c for c in f.columns if c.startswith(LEVEL_PREFIXES) and c not in keep and "_rel_" not in c]

# ---- formerly src/features_reservoir.py ----
# Reservoir-state features: fill fraction, weekly inflow volume, projected fill, water-value trend.
#
# Why: the error analysis showed that half of the hours have a clearly negative margin, yet 7% of them are ON
# (forced running: full reservoir, spill, min flow). Fill fraction and inflow-vs-free-capacity describe that
# pressure directly. max_vol per reservoir is read from data/extended/Tokke_Vinje_topology.yaml (reservoir section).

H = 168
# Mm3, "max_vol" of each reservoir in Tokke_Vinje_topology.yaml
MAX_VOL = {"Bandak": 174.2, "Vinjevatn": 9.36, "Byrtevatn": 76.71, "Botnedalsvatn": 54.17, "Kjelavatn": 84.5,
           "Staavatn": 48.0, "Venemo": 23.0, "Langesae": 78.0, "Hyljelihyl": 0.26, "Bordalsvatn": 184.2,
           "Foersvatn": 122.3, "Songavatn": 640.0, "Bitdalsvatn": 110.0, "Totak": 257.8, "Vaamarvatn": 26.2,
           "Langeidvatn": 32.1, "Vatjern": 0.3}

def add_reservoir_features(f):
    n = len(f) // H
    out = {}
    for res, vmax in MAX_VOL.items():
        vol = f[f"vol_{res}"].to_numpy(dtype=np.float32)
        inflow = f[f"inflow_{res}"].to_numpy(dtype=np.float32).reshape(n, H)
        wk_mm3 = np.repeat(inflow.sum(1) * 3600 / 1e6, H)                  # m3/s summed over the week -> Mm3
        out[f"fill_{res}"] = vol / vmax                                       # fraction of max volume at case start
        out[f"inflow_wk_{res}"] = wk_mm3
        out[f"fill_proj_{res}"] = (vol + wk_mm3) / vmax                       # fill at week end if nothing is produced
        wv0 = f[f"wv_{res}"].to_numpy(dtype=np.float32)
        wv1 = f[f"wv_end_{res}"].to_numpy(dtype=np.float32)
        out[f"wv_trend_{res}"] = wv1 - wv0                                    # optimiser's view of scarcity over the week
        out[f"wv_trend_rel_{res}"] = (wv1 - wv0) / (np.abs(wv0) + 1.0)
    for plant, res in UPSTREAM.items():                                      # 24 h window mean of the inflow feeding each plant
        inflow = f[f"inflow_{res}"].to_numpy(dtype=np.float32).reshape(n, H)
        out[f"inflow_d24_{plant}"] = np.repeat(inflow.reshape(n, 7, 24).mean(2), 24, axis=1).ravel()
    return pd.concat([f.reset_index(drop=True), pd.DataFrame(out)], axis=1)


In [ ]:
# run from the repo root so that data/, results/ and submissions/ resolve (notebooks live in notebooks/)
import os, sys, re, json, time
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

from kaggle_metric import score                                   # official metric

DEVICE = "cuda" if os.environ.get("XGB_DEVICE", "cpu") == "cuda" else "cpu"   # set XGB_DEVICE=cuda on a GPU machine
PARAMS = dict(learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, min_child_weight=5)
print("xgboost", xgb.__version__, "| device", DEVICE, "| seed", SEED)

## 1. Data

Kernel data: targets (one row per case, 14 units × 168 hourly columns), hourly day-ahead price, hourly inflow per reservoir,
daily reservoir volume and daily synthetic water value. Extended data: minimum river flow and minimum reservoir volume (hourly).

The test set (`prediction_mapping.csv`) is 725 cases, one per day from 2023-01-01 to 2024-12-25. All inputs for those weeks are
given; only the labels are hidden.

In [ ]:
uc, price, inflow, volume, water_value = load("data/kernel")
minflow, minvol = load_constraints("data/extended")
test_map = pd.read_csv("prediction_mapping.csv", parse_dates=["starttime"])
sample = pd.read_csv("sample_submission.csv", nrows=1)

label_cols = [c for c in sample.columns if c != "Run No"]
units = list(dict.fromkeys(re.match(r"result_committed_(.+)_t\d+$", c).group(1) for c in label_cols))
N_UNITS = len(units)

# targets: (cases, units*H) wide  ->  (cases*H, units) long, so row k = (case k // H, hour k % H)
Y = uc[label_cols].to_numpy(dtype=np.int8).reshape(len(uc), N_UNITS, H).transpose(0, 2, 1).reshape(-1, N_UNITS)
starts = uc["starttime"]

print(f"train cases {len(uc)} ({starts.min().date()} .. {starts.max().date()}), test cases {len(test_map)}")
print(f"targets long format {Y.shape}, mean ON rate {Y.mean():.3f}")
print("units:", units)

## 2. Why scale-free features: the price level changes every year

The mean day-ahead price moves from 9 EUR/MWh (2020) to 211 EUR/MWh (2022), and water values follow. A margin of +2 EUR is a
clear "produce" signal in 2020 and noise in 2022. Trees split on absolute thresholds, so a model trained on one price regime
transfers badly to another. The test years 2023 and 2024 are yet another regime, so this matters for the leaderboard.

In [ ]:
yearly = price.groupby(price.index.year).agg(mean_price="mean", std_price="std").round(1)
yearly["mean_wv_Vinjevatn"] = water_value["Vinjevatn"].groupby(water_value.index.year).mean().round(1)
yearly.loc[2015:2024]

Rolling validation of earlier feature sets (rounds chosen on the year before each report year, one-week gap, same XGBoost
parameters) shows the effect directly. The hard year is 2020, and removing the absolute levels helps on every year:

| feature set | features | 2020 | 2021 | 2022 | mean |
|---|---:|---:|---:|---:|---:|
| margin + constraints + context (absolute) | 268 | 0.9218 | 0.9794 | 0.9806 | 0.9606 |
| + relative + reservoir, levels kept | 407 | 0.9598 | 0.9856 | 0.9868 | 0.9774 |
| + relative + reservoir, **levels dropped** | 321 | 0.9721 | 0.9866 | 0.9871 | 0.9819 |
| same, context recomputed on the scale-free margin (**this notebook**) | 321 | **0.9737** | **0.9873** | 0.9870 | **0.9827** |

(From `src/feature_experiments.py`, rows in `results/fe_*.csv`; see `NOTES.md` for the full ablation.)

## 3. Feature construction

One function, applied to the training and the test case start times.

* `FeatureBuilder` (`src/train_xgb_baseline.py`): calendar (`t`, hour, weekday, month, day of year), hourly price and weekly
  price mean / std / min / max / rank, price of the previous and next hour, 18 hourly inflow series, 17 reservoir volumes at case
  start, 17 water values at case start and at case end (D+7).
* `add_margin_features` (`src/features_margin.py`): for each of the 9 plants, `price − wv` of the reservoir directly upstream
  (mapping read from the topology yaml), at case start, at case end, and weekly mean price minus end water value.
* `add_constraint_features` (`src/features_constraints.py`): minimum river flow at hour t and its weekly maximum (6 rivers);
  minimum reservoir volume, slack between start volume and the minimum, and the weekly worst slack (3 reservoirs).
* `add_relative_features` (`src/features_relative.py`): per plant, `price / wv` (start, end, weekly mean), `margin_end / (weekly
  price std + 1)` = `margin_end_rel`, and the share of profitable hours in the week.
* `add_reservoir_features` (`src/features_reservoir.py`): per reservoir, fill fraction `vol / max_vol` (max volume from the
  topology yaml), weekly inflow in Mm³, projected fill at week end, water-value trend over the week; 24 h inflow mean per plant.
* `level_columns` lists every absolute `price*`, `wv_*`, `margin_*` column (keeping `price_rank`, `price_week_std` and the `_rel_` columns). These are dropped.
* `add_context_features` (`src/features_context.py`) on `margin_end_rel`: centred rolling mean / max / min over 3, 6, 12 h,
  profitable hours in the same day, rank within the day, and run lengths of consecutive profitable (unprofitable) hours up to t and from t on.

In [ ]:
build_base = FeatureBuilder(price, inflow, volume, water_value)
constraints = ConstraintFeatures(minflow, minvol, volume)

def build_features(start_times, verbose=False):
    f = build_base(start_times)                                       # calendar, price, inflow, volume, water values
    f = add_margin_features(f)                                        # price - upstream water value, per plant
    f = add_constraint_features(f, start_times, constraints)          # min flow / min volume
    n_base = f.shape[1]
    f = add_reservoir_features(add_relative_features(f))              # scale-free margin + reservoir state
    n_scalefree = f.shape[1]
    levels = level_columns(f)
    f = f.drop(columns=levels)                                        # no absolute price / water-value / margin levels
    f = add_context_features(f, col="margin_end_rel")                 # temporal context of the scale-free margin
    if verbose:
        print(f"base {n_base} -> + scale-free {n_scalefree} -> drop {len(levels)} level columns -> + context = {f.shape[1]} features")
    return f

t0 = time.time()
X = build_features(starts, verbose=True)
X_test = build_features(test_map["starttime"])
assert list(X.columns) == list(X_test.columns) and len(X) == len(Y)
assert not X.isna().any().any() and not X_test.isna().any().any()
print(f"X {X.shape}, X_test {X_test.shape}, built in {time.time() - t0:.0f}s")

In [ ]:
# the absolute levels are really gone; what remains is scale-free, calendar, inflow, volume and constraint columns
assert not any(c.startswith(("price_", "wv_", "margin_")) and "_rel_" not in c and c not in ("price_rank", "price_week_std") for c in X.columns)
groups = {"calendar": ("t", "hour", "dow", "month", "doy"), "price (scale-free)": ("price_rank", "price_week_std"),
          "inflow": ("inflow_",), "volume": ("vol_",), "constraints": ("minflow_", "minvol_", "volgap_"),
          "relative margin": ("ratio_", "margin_end_rel_", "poshours_wk_"), "reservoir state": ("fill_", "inflow_wk_", "wv_trend_", "inflow_d24_"),
          "context of margin_end_rel": ("ctx_",)}
counts = {g: sum(c.startswith(p) if not p in ("t", "hour", "dow", "month", "doy", "price_rank", "price_week_std") else c == p
                 for c in X.columns for p in pre) for g, pre in groups.items()}
pd.Series(counts, name="columns").to_frame().T

In [ ]:
# the key feature for one plant and one case week: scale-free margin vs the unit's ON/OFF label
case, j = 2500, units.index("Tokke_G1")
rows = slice(case * H, (case + 1) * H)
pd.DataFrame({"hour": np.arange(H), "margin_end_rel_Tokke": X["margin_end_rel_Tokke"].to_numpy()[rows].round(2),
              "ctx_mean12_Tokke": X["ctx_mean12_Tokke"].to_numpy()[rows].round(2),
              "posrun_ahead": X["ctx_posrun_ahead_Tokke"].to_numpy()[rows], "Tokke_G1 ON": Y[rows, j]}).iloc[::12].set_index("hour").T

## 4. Honest time-aware validation

Cases are daily rolling windows, so neighbouring cases share 6/7 of their hours: random splits would leak. We split by year with a
one-week gap, and we never use the year we report on to make a modelling decision:

1. train on cases before 2021, early-stop on 2021 → number of boosting rounds;
2. train on cases before 2022 with that many rounds → **2022 micro ROC-AUC** (the reported score);
3. refit on all cases with the same number of rounds → test predictions.

(`honest_fit` in `src/train_xgb_baseline.py`.) The XGBoost parameters are plain defaults for a tabular problem of this size
(learning rate 0.05, depth 6, 80 % row and column subsampling, min child weight 5); a sweep under rolling validation moved the
mean score by less than ±0.001, so parameters are not the lever here.

In [ ]:
t0 = time.time()
res = honest_fit(X, Y, starts, PARAMS, tune_year=2021, report_year=2022, device=DEVICE)
print(f"rounds chosen on 2021: {res['n_rounds']}   2021 AUC {res['tune_auc']:.4f}   2022 report micro AUC {res['report_auc']:.4f}   [{time.time() - t0:.0f}s]")

In [ ]:
# the reported number is identical to the Kaggle metric applied to the 2022 cases in submission format
_, val_cases = split_masks(starts, 2022)
run_ids = uc.loc[val_cases, "Run No"].to_numpy()
wide = res["pred"].reshape(len(run_ids), H, N_UNITS).transpose(0, 2, 1).reshape(len(run_ids), -1)
val_submission = pd.DataFrame(wide, columns=label_cols); val_submission.insert(0, "Run No", run_ids)
val_solution = uc.loc[val_cases, ["Run No", *label_cols]]
print(f"kaggle_metric.score on 2022: {score(val_solution, val_submission):.4f}")

In [ ]:
per_unit = pd.Series(res["per_unit"], index=units, name="AUC 2022").sort_values()
per_unit.round(4).to_frame().T

The two weakest units are Lio_G1 and Haukeli_G1. Both are small plants where the optimiser sometimes runs the unit at a negative
margin (minimum-flow requirement for Lio, a tiny 0.3 Mm³ reservoir for Haukeli that must be emptied when inflow arrives). The
constraint and reservoir features recover part of that, the rest is the remaining error.

## 5. What the model uses

Gain-based importance of the stage-2 model (summed over the 14 boosters).

In [ ]:
imp = pd.Series(res["model"].get_booster().get_score(importance_type="gain"), name="gain").sort_values(ascending=False)
top = imp.head(15).round(0).to_frame()
top["group"] = ["context of scale-free margin" if c.startswith("ctx_") else "scale-free margin" if "_rel_" in c or c.startswith(("ratio_", "poshours_wk_"))
                else "reservoir state" if c.startswith(("fill_", "inflow_wk_", "wv_trend_")) else "other" for c in top.index]
top

The top of the list is the rolling context of the scale-free margin for the small plants (Kjela, Byrte, Songa, Vesle Kjela) and
the scale-free end-of-week margin of Tokke, the largest plant. Calendar, raw inflow and raw volume columns are far down: the
decision really is "price relative to water value, and for how many hours around t".

## 6. Final model and submission

Refit on every training case with the number of rounds chosen on 2021, predict the 725 test cases, and write the file in the
exact column and row order of `sample_submission.csv`.

In [ ]:
t0 = time.time()
final = xgb.XGBClassifier(n_estimators=res["n_rounds"], tree_method="hist", device=DEVICE, random_state=SEED, **PARAMS).fit(X, Y)
pred = final.predict_proba(X_test)                                                    # (725*168, 14)
wide = pred.reshape(len(test_map), H, N_UNITS).transpose(0, 2, 1).reshape(len(test_map), -1)
submission = pd.DataFrame(wide, columns=label_cols); submission.insert(0, "Run No", test_map["Run No"].to_numpy())

assert list(submission.columns) == list(sample.columns)
assert (submission["Run No"].to_numpy() == test_map["Run No"].to_numpy()).all()
assert ((wide >= 0) & (wide <= 1)).all() and not np.isnan(wide).any()

os.makedirs("submissions", exist_ok=True)
fname = f"submissions/{pd.Timestamp.today():%Y-%m-%d}_xgb_relctx_nolevel_cv{res['report_auc']:.4f}.csv"
submission.to_csv(fname, index=False)
print(f"wrote {fname}  shape {submission.shape}  mean prediction {wide.mean():.3f}  [{time.time() - t0:.0f}s]")
submission.iloc[:3, :6]

## 7. Summary

| | |
|---|---|
| model | XGBoost, one booster per unit, 14 units, rows = (case, hour) |
| features | 321, all scale-free with respect to the price level |
| validation | rounds on 2021, score on 2022, one-week gap, refit on all years |
| 2022 micro ROC-AUC | 0.9870 |
| Kaggle public (2023) | 0.99106 |

What mattered, in order of gain: price minus water value per plant (+0.14 over calendar/price/inflow features), the temporal
context of that margin (+0.002), and making it scale-free and dropping the absolute levels (+0.007 on 2021/2022, +0.05 on the
hard year 2020). Parameter tuning, stacking a second stage on the first-stage predictions, averaging predictions across the
overlapping cases of the same calendar hour, and lower learning rates with more rounds were all tried and gave nothing or hurt
(see `NOTES.md`).